MovieLens Dataset Exploration

In [2]:
import pandas as pd
from pathlib import Path

Locate the dataset

In [3]:
DATA_PATH = Path("../../data/")

movies_path = DATA_PATH / "movies.csv"
ratings_path = DATA_PATH / "ratings.csv"

print("Movies file exists:", movies_path.exists())
print("Ratings file exists:", ratings_path.exists())

Movies file exists: True
Ratings file exists: True


Load the datasets

In [4]:
# Load movie information
movies_df = pd.read_csv(movies_path)

# Load user ratings
ratings_df = pd.read_csv(ratings_path)

print("Movies Dataset:")
display(movies_df.head())

print("Ratings Dataset:")
display(ratings_df.head())

Movies Dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Ratings Dataset:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Dataset dimensions

In [5]:
print("Movies shape:", movies_df.shape)
print("Ratings shape:", ratings_df.shape)

Movies shape: (9742, 3)
Ratings shape: (100836, 4)


Column information

In [6]:
print("Movies information:")
movies_df.info()

print("\nRatings information:")
ratings_df.info()

Movies information:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB

Ratings information:
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Statistical summary

In [7]:
print("Movies summary:")
display(movies_df.describe(include="all"))

print("Ratings summary:")
display(ratings_df.describe())

Movies summary:


,movieId,title,genres
count,9742.000000,9742,9742
unique,NaN,9737,951
top,NaN,Emma (1996),Drama
freq,NaN,2,1053
mean,42200.353623,NaN,NaN
std,52160.494854,NaN,NaN
min,1.000000,NaN,NaN
25%,3248.250000,NaN,NaN
50%,7300.000000,NaN,NaN
75%,76232.000000,NaN,NaN


Ratings summary:


,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


Count unique users and movies

In [8]:
total_users = ratings_df["userId"].nunique()
total_movies = movies_df["movieId"].nunique()
total_ratings = len(ratings_df)

print("Total users:", total_users)
print("Total movies:", total_movies)
print("Total ratings:", total_ratings)

Total users: 610
Total movies: 9742
Total ratings: 100836


Understand rating distribution

In [9]:
print(ratings_df["rating"].value_counts().sort_index())

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


Calculate movie statistics

In [10]:
movie_stats=ratings_df.groupby("movieId").agg(average_rating=("rating","mean"),rating_count=("rating","count")).reset_index()
movie_stats.head()

,movieId,average_rating,rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49


- C = Average rating across all movies
- m = Minimum rating count (90th percentile)

In [11]:
C=ratings_df["rating"].mean()
m=movie_stats["rating_count"].quantile(0.90)

print("Overall average rating:", C)
print("Minimum rating count:", m)

Overall average rating: 3.501556983616962
Minimum rating count: 27.0


Calculate weighted rating

In [12]:
# Calculate weighted rating for every movie
movie_stats["weighted_rating"] = (
    (movie_stats["rating_count"] / (movie_stats["rating_count"] + m))
    * movie_stats["average_rating"]
    +
    (m / (movie_stats["rating_count"] + m))
    * C
)

movie_stats.head()

,movieId,average_rating,rating_count,weighted_rating
0,1,3.920930,215,3.874141
1,2,3.431818,110,3.445562
2,3,3.259615,52,3.342304
3,4,2.357143,7,3.265942
4,5,3.071429,49,3.224237


Merge movie information

In [13]:
popular_movies=movies_df.merge(movie_stats,on="movieId",how="inner")
popular_movies.head()

,movieId,title,genres,average_rating,rating_count,weighted_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215,3.874141
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110,3.445562
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52,3.342304
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7,3.265942
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49,3.224237


Generate recommendations

In [14]:
# Recommend movies with sufficient ratings

popular_movies = popular_movies[
    popular_movies["rating_count"] >= m
]

# Sort by weighted rating
popular_movies=popular_movies.sort_values(by="weighted_rating",ascending=False)

# Display Top 10 recommendations

top_10_movies = popular_movies[
    ["title", "genres", "average_rating", "rating_count", "weighted_rating"]
].head(10)

top_10_movies

,title,genres,average_rating,rating_count,weighted_rating
277,"Shawshank Redemption, The (1994)",Crime|Drama,4.429022,317,4.356227
659,"Godfather, The (1972)",Crime|Drama,4.289062,192,4.191973
2224,Fight Club (1999),Action|Crime|Drama|Thriller,4.272936,218,4.187927
224,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.231076,251,4.160223
46,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.237745,204,4.151697
461,Schindler's List (1993),Drama|War,4.225000,220,4.145919
257,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.197068,307,4.140844
897,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.215640,211,4.134630
1938,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.192446,278,4.131285
921,"Godfather: Part II, The (1974)",Crime|Drama,4.259690,129,4.128475


Tags Dataset

In [15]:
tags = pd.read_csv(DATA_PATH / "tags.csv")

print("TAGS DATASET")
print(tags.head())
print(tags.info())
print("Shape:", tags.shape)
print("Columns:", tags.columns.tolist())

TAGS DATASET
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable  1445714996
2       2    60756     will ferrell  1445714992
3       2    89774     Boxing story  1445715207
4       2    89774              MMA  1445715200
<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 115.2 KB
None
Shape: (3683, 4)
Columns: ['userId', 'movieId', 'tag', 'timestamp']


User-Based Collaborative Filtering

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine=create_engine(os.getenv("DATABASE_URL"))
ratings_df=pd.read_sql(
    """
    SELECT dataset_user_id, movie_id, rating
    FROM ratings
    WHERE dataset_user_id IS NOT NULL
    """,
    engine
)

ratings_df.head()

,dataset_user_id,movie_id,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


Interaction matrix

In [17]:
user_item_matrix=ratings_df.pivot_table(
    index="dataset_user_id", #row
    columns="movie_id", #col
    values="rating"
)
user_item_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
dataset_user_id,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
print("Matrix shape:", user_item_matrix.shape)
print("Total users:", user_item_matrix.shape[0])
print("Total movies:", user_item_matrix.shape[1])
print("Missing values:", user_item_matrix.isnull().sum().sum())

Matrix shape: (610, 9724)
Total users: 610
Total movies: 9724
Missing values: 5830804


In [19]:
#Replacing missing ratings with 0 for cosine similarity
user_item_filled=user_item_matrix.fillna(0)
user_item_filled.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
dataset_user_id,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Calculate user similarity

In [20]:
user_similarity=cosine_similarity(user_item_filled)

user_similarity_df=pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
user_similarity_df.head()

dataset_user_id,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
dataset_user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


Finding similar users

In [21]:
user_id=1

similar_users=user_similarity_df[user_id].sort_values(ascending=False)

#Exclude user themselves
similar_users=similar_users.drop(user_id)
similar_users.head(5)

dataset_user_id
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
Name: 1, dtype: float64

Generate recommendations

In [22]:
def recommend_movies(user_id,top_n=10,similar_users_count=10):
    #get similar users
    similar_users=user_similarity_df[user_id].sort_values(ascending=False).drop(user_id)
    similar_users=similar_users.head(similar_users_count)
    
    #get movies the target user hasn't rated
    user_ratings=user_item_matrix.loc[user_id]
    
    unrated_movies=user_ratings[user_ratings.isna()].index
    
    #collect recommendations from similar users
    recommendations={}
    
    for similar_user_id,similarity_score in similar_users.items():
        # Get similar user's ratings
        ratings=user_item_matrix.loc[similar_user_id]
        
        for movie_id in unrated_movies:
            rating=ratings[movie_id]
            
            if pd.notna(rating) and rating>=4:
                if movie_id not in recommendations:
                    recommendations[movie_id]=0
                recommendations[movie_id]+=similarity_score* rating
                
    #sort recommendations
    recommendations=sorted(recommendations.items(),key=lambda x:x[1], reverse=True)
    
    return recommendations[:top_n]

In [23]:
recommendations = recommend_movies(user_id=1, top_n=10)

recommendations

[(589, np.float64(13.146917710802267)),
 (858, np.float64(13.134549527466067)),
 (1200, np.float64(12.936063266381074)),
 (2762, np.float64(12.432916270630933)),
 (1221, np.float64(12.104994580853713)),
 (541, np.float64(11.55601355118029)),
 (1036, np.float64(10.88940948820309)),
 (1653, np.float64(10.561736783572758)),
 (1610, np.float64(10.342091134483228)),
 (32, np.float64(10.332388535018081))]

Display movie titles

In [24]:
recommended_movie_ids = [
    movie_id for movie_id, score in recommendations
]

recommended_movies=pd.read_sql(
    """
    SELECT movie_id,title,genres
    FROM movies
    """,
    engine
)

recommended_movies=recommended_movies[recommended_movies["movie_id"].isin(recommended_movie_ids)]
recommended_movies

,movie_id,title,genres
26,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
467,541,Blade Runner (1982),Action|Sci-Fi|Thriller
498,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
649,858,"Godfather, The (1972)",Crime|Drama
780,1036,Die Hard (1988),Action|Crime|Thriller
888,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi
908,1221,"Godfather: Part II, The (1974)",Crime|Drama
1196,1610,"Hunt for Red October, The (1990)",Action|Adventure|Thriller
1228,1653,Gattaca (1997),Drama|Sci-Fi|Thriller
2051,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery


Item-Based Collaborative Filtering (IBCF)

In [25]:
user_item_matrix.shape

(610, 9724)

In [26]:
item_user_matrix=user_item_matrix.T #user matrix-col, movie-row
item_user_matrix.shape

(9724, 610)

calculate movie similarity

In [27]:
# Replace missing values with 0
item_user_filled = item_user_matrix.fillna(0)

# Calculate similarity between movies
item_similarity = cosine_similarity(item_user_filled)

# Create similarity DataFrame
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

item_similarity_df.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.410562,0.296917,0.035573,0.308762,0.376316,0.277491,0.131629,0.232586,0.395573,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.410562,1.000000,0.282438,0.106415,0.287795,0.297009,0.228576,0.172498,0.044835,0.417693,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.296917,0.282438,1.000000,0.092406,0.417802,0.284257,0.402831,0.313434,0.304840,0.242954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.035573,0.106415,0.092406,1.000000,0.188376,0.089685,0.275035,0.158022,0.000000,0.095598,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.308762,0.287795,0.417802,0.188376,1.000000,0.298969,0.474002,0.283523,0.335058,0.218061,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Find movies similar to a particular movie

In [28]:
movie_id=589

similar_movies=item_similarity_df[movie_id].sort_values(ascending=False)

#Remove the movie itself
similar_movies=similar_movies.drop(movie_id)
similar_movies.head(10)

movie_id
480     0.719983
1240    0.695724
110     0.659827
592     0.645603
457     0.637561
377     0.630092
1196    0.618530
380     0.611164
296     0.610284
356     0.600886
Name: 589, dtype: float64

Recommend movies

In [29]:
def recommend_movies_item_based(user_id, top_n=10):

    # Get movies already rated by the user
    user_ratings = user_item_matrix.loc[user_id].dropna()

    # Consider movies rated 4 or higher
    liked_movies = user_ratings[user_ratings >= 4]

    recommendations = {}

    # Find similar movies for each liked movie
    for movie_id, rating in liked_movies.items():

        similar_movies = (
            item_similarity_df[movie_id]
            .sort_values(ascending=False)
            .drop(movie_id)
            .head(20)
        )

        for similar_movie_id, similarity_score in similar_movies.items():

            # Skip movies already rated
            if similar_movie_id in user_ratings.index:
                continue

            if similarity_score <= 0:
                continue

            if similar_movie_id not in recommendations:
                recommendations[similar_movie_id] = [0, 0]

            # Accumulate weighted rating
            recommendations[similar_movie_id][0] += (
                similarity_score * rating
            )

            # Accumulate similarity
            recommendations[similar_movie_id][1] += similarity_score

    # Calculate predicted ratings
    predicted_ratings = {}

    for movie_id, values in recommendations.items():

        weighted_sum = values[0]
        similarity_sum = values[1]

        predicted_rating = weighted_sum / similarity_sum

        predicted_ratings[movie_id] = predicted_rating

    # Sort recommendations by predicted rating
    sorted_recommendations = sorted(
        predicted_ratings.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Return top recommendations
    return sorted_recommendations[:top_n]

In [30]:
item_recommendations = recommend_movies_item_based(
    user_id=1,
    top_n=10
)

item_recommendations

[(778, 5.000000000000001),
 (4275, 5.000000000000001),
 (585, 5.000000000000001),
 (276, 5.000000000000001),
 (196, 5.000000000000001),
 (908, 5.000000000000001),
 (31116, 5.000000000000001),
 (4710, 5.000000000000001),
 (2097, 5.000000000000001),
 (5548, 5.000000000000001)]

Display movie titles

In [31]:
# Get all movie information
all_movies = pd.read_sql(
    """
    SELECT movie_id, title, genres
    FROM movies
    """,
    engine
)

# Convert recommendations into DataFrame
item_recommendations_df = pd.DataFrame(
    item_recommendations,
    columns=["movie_id", "predicted_rating"]
)

# Merge with movie information
item_recommendations_df = item_recommendations_df.merge(
    all_movies,
    on="movie_id",
    how="left"
)

# Display recommendations
item_recommendations_df[
    ["movie_id", "title", "genres", "predicted_rating"]
]

,movie_id,title,genres,predicted_rating
0,778,Trainspotting (1996),Comedy|Crime|Drama,5.0
1,4275,Krull (1983),Action|Adventure|Fantasy|Sci-Fi,5.0
2,585,"Brady Bunch Movie, The (1995)",Comedy,5.0
3,276,Milk Money (1994),Comedy|Romance,5.0
4,196,Species (1995),Horror|Sci-Fi,5.0
5,908,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller,5.0
6,31116,Sergeant York (1941),Drama|War,5.0
7,4710,"Shootist, The (1976)",Drama|Western,5.0
8,2097,Something Wicked This Way Comes (1983),Children|Drama|Fantasy|Mystery|Thriller,5.0
9,5548,Down and Out in Beverly Hills (1986),Comedy,5.0


Content-Based Filtering

In [32]:
all_movies.head()

,movie_id,title,genres
0,2,Jumanji (1995),Adventure|Children|Fantasy
1,3,Grumpier Old Men (1995),Comedy|Romance
2,4,Waiting to Exhale (1995),Comedy|Drama|Romance
3,5,Father of the Bride Part II (1995),Comedy
4,7,Sabrina (1995),Comedy|Romance


Convert genres into numerical features

In [34]:
from sklearn.preprocessing import MultiLabelBinarizer

#Handle missing genres
all_movies["genres"]=all_movies["genres"].fillna("")

#convert genre strings into lists
genre_lists=all_movies["genres"].apply(lambda x:x.split("|") if x else [])

#convert genres into binary features
mlb=MultiLabelBinarizer()

genre_matrix=mlb.fit_transform(genre_lists)

#Create DF
genre_df=pd.DataFrame(
    genre_matrix,
    index=all_movies["movie_id"],
    columns=mlb.classes_
)

genre_df.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movie_id,,,,,,,,,,,,,,,,,,,,
2,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0


Calculating movie similarity

In [35]:
content_similarity=cosine_similarity(genre_matrix)

content_similarity_df=pd.DataFrame(
    content_similarity,
    index=all_movies["movie_id"],
    columns=all_movies["movie_id"]
)

content_similarity_df.head()

movie_id,2,3,4,5,7,8,9,10,12,13,...,142509,440,954,1171,1608,1810,3386,6535,8011,11
movie_id,,,,,,,,,,,,,,,,,,,,,
2,1.0,0.000000,0.000000,0.000000,0.000000,0.816497,0.0,0.333333,0.000000,0.666667,...,0.333333,0.000000,0.00000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000
3,0.0,1.000000,0.816497,0.707107,1.000000,0.000000,0.0,0.000000,0.500000,0.000000,...,0.000000,1.000000,0.00000,0.707107,0.0,0.500000,0.000000,0.707107,0.0,0.816497
4,0.0,0.816497,1.000000,0.577350,0.816497,0.000000,0.0,0.000000,0.408248,0.000000,...,0.000000,0.816497,0.57735,0.577350,0.0,0.816497,0.333333,0.577350,0.0,1.000000
5,0.0,0.707107,0.577350,1.000000,0.707107,0.000000,0.0,0.000000,0.707107,0.000000,...,0.000000,0.707107,0.00000,1.000000,0.0,0.707107,0.000000,1.000000,0.0,0.577350
7,0.0,1.000000,0.816497,0.707107,1.000000,0.000000,0.0,0.000000,0.500000,0.000000,...,0.000000,1.000000,0.00000,0.707107,0.0,0.500000,0.000000,0.707107,0.0,0.816497


Find similar movies

In [36]:
movie_id = 589

similar_movies=(
    content_similarity_df[movie_id]
    .sort_values(ascending=False)
    .drop(movie_id)
    .head(10)
)
similar_movies

movie_id
1373      1.0
26819     1.0
26985     1.0
8677      1.0
189547    1.0
168252    1.0
2641      1.0
60040     1.0
1676      1.0
64997     1.0
Name: 589, dtype: float64

In [38]:
content_results = pd.DataFrame(
    similar_movies.items(),
    columns=["movie_id", "similarity"]
)

content_results = content_results.merge(
    all_movies,
    on="movie_id",
    how="left"
)

content_results[
    ["movie_id", "title", "genres", "similarity"]
]

,movie_id,title,genres,similarity
0,1373,Star Trek V: The Final Frontier (1989),Action|Sci-Fi,1.0
1,26819,Fortress (1992),Action|Sci-Fi,1.0
2,26985,Nirvana (1997),Action|Sci-Fi,1.0
3,8677,Flash Gordon Conquers the Universe (1940),Action|Sci-Fi,1.0
4,189547,Iron Soldier (2010),Action|Sci-Fi,1.0
5,168252,Logan (2017),Action|Sci-Fi,1.0
6,2641,Superman II (1980),Action|Sci-Fi,1.0
7,60040,"Incredible Hulk, The (2008)",Action|Sci-Fi,1.0
8,1676,Starship Troopers (1997),Action|Sci-Fi,1.0
9,64997,War of the Worlds (2005),Action|Sci-Fi,1.0


Recommendation

In [39]:
def recommend_movies_content_based(user_id, top_n=10):
    #get movies rated by the user
    user_ratings=user_item_matrix.loc[user_id].dropna()
    
    #get movies the user likes
    liked_movies=user_ratings[user_ratings>=4]
    
    recommendations={}
    
    # Find movies similar to each liked movie
    for movie_id,rating in liked_movies.items():
        similar_movies=(
            content_similarity_df[movie_id]
            .sort_values(ascending=False)
            .drop(movie_id)
            .head(20)
        )
        
        for similar_movie_id,similarity_score in similar_movies.items():
            # Skip movies already rated
            if similar_movie_id in user_ratings.index:
                continue

            if similarity_score <= 0:
                continue

            if similar_movie_id not in recommendations:
                recommendations[similar_movie_id] = [0, 0]

            recommendations[similar_movie_id][0] += (
                similarity_score * rating
            )

            recommendations[similar_movie_id][1] += similarity_score

    # Calculate predicted ratings
    predicted_ratings = {}

    for movie_id, values in recommendations.items():

        predicted_ratings[movie_id] = (
            values[0] / values[1]
        )

    # Sort recommendations
    sorted_recommendations = sorted(
        predicted_ratings.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return sorted_recommendations[:top_n]

In [40]:
content_recommendations = recommend_movies_content_based(
    user_id=1,
    top_n=10
)

content_recommendations

[(31364, 5.000000000000001),
 (72129, 5.000000000000001),
 (33830, 5.000000000000001),
 (2733, 5.000000000000001),
 (88108, 5.000000000000001),
 (32296, 5.000000000000001),
 (71970, 5.000000000000001),
 (43836, 5.000000000000001),
 (31925, 5.000000000000001),
 (4027, 5.000000000000001)]

In [41]:
content_recommendations_df = pd.DataFrame(
    content_recommendations,
    columns=["movie_id", "predicted_rating"]
)

content_recommendations_df = content_recommendations_df.merge(
    all_movies,
    on="movie_id",
    how="left"
)

content_recommendations_df[
    ["movie_id", "title", "genres", "predicted_rating"]
]

,movie_id,title,genres,predicted_rating
0,31364,Memories of Murder (Salinui chueok) (2003),Crime|Drama|Mystery|Thriller,5.0
1,72129,Saw VI (2009),Crime|Horror|Mystery|Thriller,5.0
2,33830,Herbie: Fully Loaded (2005),Adventure|Comedy|Romance,5.0
3,2733,Vibes (1988),Adventure|Comedy|Romance,5.0
4,88108,Monte Carlo (2011),Adventure|Comedy|Romance,5.0
5,32296,Miss Congeniality 2: Armed and Fabulous (2005),Adventure|Comedy|Crime,5.0
6,71970,"Princess and the Pirate, The (1944)",Adventure|Comedy|Romance,5.0
7,43836,"Pink Panther, The (2006)",Adventure|Comedy|Crime,5.0
8,31925,Royal Flash (1975),Adventure|Comedy|Romance,5.0
9,4027,"O Brother, Where Art Thou? (2000)",Adventure|Comedy|Crime,5.0


Hybrid Recommendation System

In [43]:
def normalize_scores(recommendations):
    scores=pd.Series(
        dict(recommendations),
        dtype=float
    )
    
    if scores.empty:
        return scores
    
    min_score=scores.min()
    max_score=scores.max()
    
    if max_score==min_score:
        return pd.Series(0.5,index=scores.index)
    
    normalized_scores=((scores-min_score)/(max_score-min_score))
    return normalized_scores

Recommendations

In [44]:
user_id = 1

# Item-Based Collaborative Filtering
item_recommendations=recommend_movies_item_based(user_id=user_id,top_n=100)

# Content-Based Filtering
content_recommendations=recommend_movies_content_based(user_id=user_id,top_n=100)

Normalized lists

In [46]:
item_scores = normalize_scores(item_recommendations)

content_scores = normalize_scores(content_recommendations)

# Get all movie IDs recommended by either algorithm
all_recommended_movies=(
    set(item_scores.index)|
    set(content_scores.index)
)

hybrid_scores={}

for movie_id in all_recommended_movies:

    item_score = item_scores.get(movie_id, 0)

    content_score = content_scores.get(movie_id, 0)

    hybrid_score = (
        0.5 * item_score +
        0.5 * content_score
    )

    hybrid_scores[movie_id] = hybrid_score

In [47]:
hybrid_recommendations = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

hybrid_recommendations = hybrid_recommendations[:10]

hybrid_recommendations

[(3078, np.float64(0.5)),
 (2599, np.float64(0.5)),
 (2097, np.float64(0.5)),
 (1078, np.float64(0.5)),
 (1079, np.float64(0.5)),
 (68159, np.float64(0.5)),
 (585, np.float64(0.5)),
 (4710, np.float64(0.5)),
 (8807, np.float64(0.5)),
 (161922, np.float64(0.5))]

In [48]:
hybrid_recommendations_df = pd.DataFrame(
    hybrid_recommendations,
    columns=["movie_id", "hybrid_score"]
)

hybrid_recommendations_df = hybrid_recommendations_df.merge(
    all_movies,
    on="movie_id",
    how="left"
)

hybrid_recommendations_df[
    ["movie_id", "title", "genres", "hybrid_score"]
]

,movie_id,title,genres,hybrid_score
0,3078,Liberty Heights (1999),Drama,0.5
1,2599,Election (1999),Comedy,0.5
2,2097,Something Wicked This Way Comes (1983),Children|Drama|Fantasy|Mystery|Thriller,0.5
3,1078,Bananas (1971),Comedy|War,0.5
4,1079,"Fish Called Wanda, A (1988)",Comedy|Crime,0.5
5,68159,State of Play (2009),Crime|Drama|Thriller,0.5
6,585,"Brady Bunch Movie, The (1995)",Comedy,0.5
7,4710,"Shootist, The (1976)",Drama|Western,0.5
8,8807,Harold and Kumar Go to White Castle (2004),Adventure|Comedy,0.5
9,161922,The Edge of Seventeen (2016),Comedy,0.5
